In [ ]:
# ================= IMPORTING LIBRARIES =================
import os, cv2, hashlib, time, pathlib
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# ================= REPRODUCIBILITY =================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)


# ================= CONFIG =================
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 25
AUTOTUNE = tf.data.AUTOTUNE


# GPU SETUP
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# ============================================================
# PRE-SPLIT CSV FILES
# ============================================================

TRAIN_CSV = r"C:\Users\ALFA\Codes\Optical_Coherence_Tomography\Training\ORIGINAL_OCTDL_DATA_REDUCED\OCT_SPLIT\train_augmented.csv"

VAL_CSV = r"C:\Users\ALFA\Codes\Optical_Coherence_Tomography\Training\ORIGINAL_OCTDL_DATA_REDUCED\OCT_SPLIT\validation.csv"

TEST_CSV = r"C:\Users\ALFA\Codes\Optical_Coherence_Tomography\Training\ORIGINAL_OCTDL_DATA_REDUCED\OCT_SPLIT\test.csv"

# ============================================================
# IMAGE DIRECTORIES
# ============================================================

TRAIN_DIR = r"C:\Users\ALFA\Codes\Optical_Coherence_Tomography\Training\ORIGINAL_OCTDL_DATA_REDUCED\OCT_SPLIT\train"

VAL_DIR = r"C:\Users\ALFA\Codes\Optical_Coherence_Tomography\Training\ORIGINAL_OCTDL_DATA_REDUCED\OCT_SPLIT\validation"

TEST_DIR = r"C:\Users\ALFA\Codes\Optical_Coherence_Tomography\Training\ORIGINAL_OCTDL_DATA_REDUCED\OCT_SPLIT\test"

# ================= LOAD PRE-SPLIT CSV FILES =================

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)


# ================= IMAGE FINDER =================

def find_image_file(base_dir, disease, file_name):

    file_name = pathlib.Path(str(file_name)).stem

    for ext in ['', '.jpg', '.png', '.jpeg', '.tif']:

        p1 = os.path.join(
            base_dir,
            str(disease),
            f"{file_name}{ext}"
        )

        p2 = os.path.join(
            base_dir,
            f"{file_name}{ext}"
        )

        if os.path.exists(p1):
            return p1

        if os.path.exists(p2):
            return p2

    return None

# ================= LOAD IMAGE PATHS =================

train_df['filepath'] = train_df.apply(
    lambda r: find_image_file(
        TRAIN_DIR,
        r['disease'],
        r['file_name']
    ),
    axis=1
)

val_df['filepath'] = val_df.apply(
    lambda r: find_image_file(
        VAL_DIR,
        r['disease'],
        r['file_name']
    ),
    axis=1
)

test_df['filepath'] = test_df.apply(
    lambda r: find_image_file(
        TEST_DIR,
        r['disease'],
        r['file_name']
    ),
    axis=1
)

# ================= REMOVE MISSING FILES =================

train_df = train_df.dropna(subset=['filepath']).reset_index(drop=True)
val_df   = val_df.dropna(subset=['filepath']).reset_index(drop=True)
test_df  = test_df.dropna(subset=['filepath']).reset_index(drop=True)

print("\n=== DATA LOADING SUMMARY ===")
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

# ================= PATIENT-LEVEL CHECK =================

print("\n=== PATIENT-LEVEL SPLIT CHECK ===")

train_patients = set(train_df['patient_id'])
val_patients   = set(val_df['patient_id'])
test_patients  = set(test_df['patient_id'])

print("Train-Val overlap :", len(train_patients & val_patients))
print("Train-Test overlap:", len(train_patients & test_patients))
print("Val-Test overlap  :", len(val_patients & test_patients))

if (
    len(train_patients & val_patients) == 0 and
    len(train_patients & test_patients) == 0 and
    len(val_patients & test_patients) == 0
):
    print("✅ Perfect patient-level split (no leakage)")
else:
    print("❌ Patient leakage detected!")

# ================= DUPLICATE CHECK =================

def hash_image(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print("\n=== DUPLICATE IMAGE CHECK ===")

combined_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

combined_df['img_hash'] = combined_df['filepath'].apply(hash_image)

duplicate_counts = combined_df['img_hash'].value_counts()
num_duplicate_images = (duplicate_counts > 1).sum()

print("Images with duplicates:", num_duplicate_images)

cross_patient_dup = combined_df.groupby('img_hash')['patient_id'].nunique()
cross_patient_dup = cross_patient_dup[cross_patient_dup > 1]

print("Cross-patient duplicate images:", len(cross_patient_dup))

if len(cross_patient_dup) > 0:
    print("⚠️ WARNING: Data leakage risk detected!")
else:
    print("✅ No cross-patient leakage from duplicate images")


# ============================================================
# REMOVE DUPLICATE IMAGES
# ============================================================

print("\n=== REMOVING DUPLICATE IMAGES ===")

# ---------------- REMOVE EXACT DUPLICATES ----------------

before_total = len(combined_df)

combined_df = combined_df.drop_duplicates(
    subset='img_hash',
    keep='first'
).reset_index(drop=True)

after_total = len(combined_df)

print(f"Removed duplicate images: {before_total - after_total}")
print(f"Remaining unique images: {after_total}")

# ---------------- REMOVE CROSS-PATIENT DUPLICATES ----------------

cross_patient_dup = combined_df.groupby('img_hash')['patient_id'].nunique()
cross_patient_hashes = cross_patient_dup[
    cross_patient_dup > 1
].index

if len(cross_patient_hashes) > 0:

    print(f"Removing cross-patient duplicates: {len(cross_patient_hashes)}")

    combined_df = combined_df[
        ~combined_df['img_hash'].isin(cross_patient_hashes)
    ].reset_index(drop=True)

else:
    print("No cross-patient duplicates found")

# ============================================================
# SPLIT BACK INTO TRAIN / VAL / TEST
# ============================================================

train_df = combined_df[
    combined_df['filepath'].str.contains(TRAIN_DIR, regex=False)
].reset_index(drop=True)

val_df = combined_df[
    combined_df['filepath'].str.contains(VAL_DIR, regex=False)
].reset_index(drop=True)

test_df = combined_df[
    combined_df['filepath'].str.contains(TEST_DIR, regex=False)
].reset_index(drop=True)

# ============================================================
# FINAL DUPLICATE CHECK
# ============================================================

print("\n=== FINAL DUPLICATE CHECK ===")

final_duplicate_count = combined_df['img_hash'].duplicated().sum()

print("Remaining duplicates:", final_duplicate_count)

if final_duplicate_count == 0:
    print("✅ All duplicate images removed successfully")
else:
    print("⚠️ Some duplicates still remain")

# ============================================================
# UPDATED DATA SUMMARY
# ============================================================

print("\n=== UPDATED DATASET SUMMARY ===")
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

# ================= ENCODING =================

disease_encoder = LabelEncoder()
severity_encoder = LabelEncoder()

combined_df['disease_encoded'] = disease_encoder.fit_transform(
    combined_df['disease']
)

combined_df['severity_encoded'] = severity_encoder.fit_transform(
    combined_df['severity_level']
)



# ================= APPLY ENCODING BACK =================

encoding_cols = [
    'file_name',
    'disease_encoded',
    'severity_encoded'
]

train_df = train_df.merge(
    combined_df[encoding_cols],
    on='file_name',
    how='left'
)

val_df = val_df.merge(
    combined_df[encoding_cols],
    on='file_name',
    how='left'
)

test_df = test_df.merge(
    combined_df[encoding_cols],
    on='file_name',
    how='left'
)



# ============================================================
# INTERNAL DUPLICATE IMAGE CHECK
# ============================================================

import hashlib

def hash_image(path):

    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

# ============================================================
# FUNCTION TO CHECK DUPLICATES
# ============================================================

def check_internal_duplicates(name, df):

    print(f"\n=== {name} INTERNAL DUPLICATE CHECK ===")

    # create image hashes
    df = df.copy()

    df['img_hash'] = df['filepath'].apply(hash_image)

    # count duplicates
    duplicate_count = df['img_hash'].duplicated().sum()

    print("Duplicate images:", duplicate_count)

    # show duplicated rows
    duplicated_rows = df[
        df['img_hash'].duplicated(keep=False)
    ].sort_values('img_hash')

    if len(duplicated_rows) > 0:

        print("\nSample duplicated entries:")

        print(
            duplicated_rows[
                ['patient_id', 'disease', 'file_name']
            ].head(10)
        )

    else:
        print("✅ No internal duplicates found")

# ============================================================
# RUN CHECKS
# ============================================================

check_internal_duplicates("TRAIN", train_df)

check_internal_duplicates("VALIDATION", val_df)

check_internal_duplicates("TEST", test_df)

# ================= OVERALL DISTRIBUTION =================

print("\n=== TRAIN DISTRIBUTION ===")
print(train_df['disease'].value_counts())

print("\n=== VALIDATION DISTRIBUTION ===")
print(val_df['disease'].value_counts())

print("\n=== TEST DISTRIBUTION ===")
print(test_df['disease'].value_counts())

# ================= BAR CHART =================

import matplotlib.pyplot as plt

train_counts = train_df['disease'].value_counts()
val_counts   = val_df['disease'].value_counts()
test_counts  = test_df['disease'].value_counts()

split_dist = pd.DataFrame({
    'Train': train_counts,
    'Validation': val_counts,
    'Test': test_counts
}).fillna(0)

ax = split_dist.plot(
    kind='bar',
    figsize=(9,6),
    width=0.9
)

plt.title(
    'Disease Distribution Across Train, Validation, and Test Sets',
    pad=15
)

plt.xlabel('Disease')
plt.ylabel('Number of Images')

plt.xticks(rotation=45)
plt.legend(title='Dataset Split')

for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=2)

plt.tight_layout()
plt.show()

# ================= FEATURE ENGINEERING =================

for df_ in [train_df, val_df, test_df]:

    df_.fillna({
        'sex':'unknown',
        'eye':'unknown'
    }, inplace=True)

    df_['age'] = 2024 - df_['year']

    df_['aspect_ratio'] = (
        df_['image_width'] /
        (df_['image_hight'] + 1e-6)
    )

# ================= SCALING =================

num_cols = [
    'image_width',
    'image_hight',
    'aspect_ratio'
]

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df[num_cols])
val_scaled   = scaler.transform(val_df[num_cols])
test_scaled  = scaler.transform(test_df[num_cols])

for i, c in enumerate(num_cols):

    train_df[f'scaled_{c}'] = train_scaled[:, i]
    val_df[f'scaled_{c}']   = val_scaled[:, i]
    test_df[f'scaled_{c}']  = test_scaled[:, i]

# ================= ENCODING =================

for c in ['sex', 'eye']:

    le = LabelEncoder()

    train_df[f'{c}_encoded'] = le.fit_transform(train_df[c])

    val_df[f'{c}_encoded'] = le.transform(val_df[c])

    test_df[f'{c}_encoded'] = le.transform(test_df[c])

feature_cols = [
    c for c in train_df.columns
    if c.startswith('scaled_') or c.endswith('_encoded')
]

train_clinical = train_df[feature_cols].values.astype(np.float32)
val_clinical   = val_df[feature_cols].values.astype(np.float32)
test_clinical  = test_df[feature_cols].values.astype(np.float32)

# ================= DATASET =================

def load_and_preprocess_image(path):

    def _read(p):

        p = p.decode()

        img = cv2.imread(
            p,
            cv2.IMREAD_GRAYSCALE
        )

        img = cv2.resize(
            img,
            IMAGE_SIZE
        )

        img = (
            img - img.mean()
        ) / (
            img.std() + 1e-6
        )

        return img[..., None].astype(np.float32)

    img = tf.numpy_function(
        _read,
        [path],
        tf.float32
    )

    img.set_shape((*IMAGE_SIZE, 1))

    return img

def make_dataset(df, clinical, shuffle=True):

    ds = tf.data.Dataset.from_tensor_slices(
        (
            (
                df['filepath'].values,
                clinical
            ),
            {
                'disease_output': df['disease_encoded'],
                'severity_output': df['severity_encoded']
            }
        )
    )

    if shuffle:
        ds = ds.shuffle(
            len(df),
            seed=RANDOM_STATE,
            reshuffle_each_iteration=True)

         # ===== DETERMINISTIC PIPELINE =====
    options = tf.data.Options()
    options.experimental_deterministic = True
    ds = ds.with_options(options)

    def map_fn(x, y):

        img = load_and_preprocess_image(x[0])

        return (img, x[1]), y

    return ds.map(map_fn).batch(BATCH_SIZE).prefetch(1)

train_ds = make_dataset(
    train_df,
    train_clinical
)

val_ds = make_dataset(
    val_df,
    val_clinical,
    False
)

test_ds = make_dataset(
    test_df,
    test_clinical,
    False
)

# ================= CBAM =================
def channel_attention(x, ratio=8):
    ch = x.shape[-1]
    avg = layers.GlobalAveragePooling2D()(x)
    maxp = layers.GlobalMaxPooling2D()(x)
    shared_1 = layers.Dense(ch//ratio, activation='relu')
    shared_2 = layers.Dense(ch, activation='sigmoid')
    avg_out = shared_2(shared_1(avg))
    max_out = shared_2(shared_1(maxp))
    scale = layers.Add()([avg_out, max_out])
    scale = layers.Reshape((1,1,ch))(scale)
    return layers.Multiply()([x, scale])

def spatial_attention(x):
    avg = tf.reduce_mean(x, axis=-1, keepdims=True)
    maxp = tf.reduce_max(x, axis=-1, keepdims=True)
    concat = layers.Concatenate()([avg, maxp])
    attn = layers.Conv2D(1, 7, padding='same', activation='sigmoid')(concat)
    return layers.Multiply()([x, attn])

def cbam_block(x):
    return spatial_attention(channel_attention(x))

# ================= CROSS MODAL =================
def cross_modal_attention(img_feat, clin_feat, dim=128, heads=4):
    img_proj = layers.Dense(dim)(img_feat)
    clin_proj = layers.Dense(dim)(clin_feat)
    img_token = layers.Reshape((1, dim))(img_proj)
    clin_token = layers.Reshape((1, dim))(clin_proj)
    clin_attn = layers.MultiHeadAttention(heads, dim//heads)(clin_token, img_token)
    img_attn  = layers.MultiHeadAttention(heads, dim//heads)(img_token, clin_token)
    return layers.Concatenate()([
        img_proj, clin_proj,
        layers.Flatten()(img_attn),
        layers.Flatten()(clin_attn)
    ])

# ================= MODEL =================
def build():
    img = Input((*IMAGE_SIZE, 1), name='input_image')
    clin = Input((len(feature_cols),), name='input_clinical')

    x = layers.Conv2D(32,3,padding='same',activation='relu')(img)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D()(x)

    x = layers.Conv2D(64,3,padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D()(x)

    x = layers.Conv2D(128,3,padding='same',activation='relu')(x)
    x = layers.MaxPool2D()(x)

    x = layers.Conv2D(256,3,padding='same',activation='relu',name='conv4_256')(x)
    x = cbam_block(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.LayerNormalization()(x)

    y = layers.Dense(64,activation='relu')(clin)
    y = layers.BatchNormalization()(y)
    y = layers.Dense(64,activation='relu')(y)
    y = layers.LayerNormalization()(y)

    z = cross_modal_attention(x, y)
    z = layers.Dense(128,activation='relu')(z)
    z = layers.Dropout(0.6)(z)

    disease = layers.Dense(len(disease_encoder.classes_),activation='softmax',name='disease_output')(z)
    severity = layers.Dense(len(severity_encoder.classes_),activation='softmax',name='severity_output')(z)

    return Model([img, clin],[disease, severity])

model = build()

model.compile(
    optimizer=keras.optimizers.Adam(1e-6),
    loss={'disease_output':'sparse_categorical_crossentropy','severity_output':'sparse_categorical_crossentropy'},
    loss_weights={'disease_output':0.1,'severity_output':0.1},
    metrics={'disease_output':['accuracy'],'severity_output':['accuracy']}
)

print("\n=== MODEL READY: CBAM + CROSS-MODAL ATTENTION ===")

# ============================================================
# SAVE BEST MODEL
# ============================================================

checkpoint = keras.callbacks.ModelCheckpoint(
        filepath='best_oct_model.keras',
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        checkpoint,
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-6)
    ]
 )


# ============================================================
# LOAD BEST SAVED MODEL
# ============================================================

best_model = keras.models.load_model(
    'best_oct_model.keras'
)

# ================= MACRO AUC =================
y_pred_disease, y_pred_severity = best_model.predict(test_ds)

macro_auc_disease = roc_auc_score(
    label_binarize(test_df['disease_encoded'], classes=np.unique(test_df['disease_encoded'])),
    y_pred_disease, average='macro', multi_class='ovr')

macro_auc_severity = roc_auc_score(
    label_binarize(test_df['severity_encoded'], classes=np.unique(test_df['severity_encoded'])),
    y_pred_severity, average='macro', multi_class='ovr')

# ================= INFERENCE TIME =================
start = time.time()
_ = best_model.predict(test_ds)
end = time.time()

# ================= FINAL METRICS =================
from sklearn.metrics import accuracy_score, cohen_kappa_score

y_pred_disease_labels = np.argmax(y_pred_disease, axis=1)
y_pred_severity_labels = np.argmax(y_pred_severity, axis=1)

# ===== INDIVIDUAL TASK METRICS =====
acc_disease = accuracy_score(test_df['disease_encoded'], y_pred_disease_labels)
acc_severity = accuracy_score(test_df['severity_encoded'], y_pred_severity_labels)

kappa_disease = cohen_kappa_score(test_df['disease_encoded'], y_pred_disease_labels)
kappa_severity = cohen_kappa_score(test_df['severity_encoded'], y_pred_severity_labels)

# ===== COMBINED (ORIGINAL BEHAVIOR - UNCHANGED) =====
acc = (acc_disease + acc_severity) / 2
kappa = (kappa_disease + kappa_severity) / 2
macro_auc = (macro_auc_disease + macro_auc_severity) / 2
inference_time = end - start
time_per_image = inference_time / len(test_df)

# ===== PRINT CLEAN RESULTS =====
print("\n================ FINAL RESULTS ================\n")

print("----- DISEASE TASK -----")
print(f"Accuracy       : {acc_disease:.3f}")
print(f"Kappa          : {kappa_disease:.3f}")
print(f"Macro AUC      : {macro_auc_disease:.3f}")
print(f"Inference Time : {inference_time:.3f} sec")
print(f"Time/Image     : {time_per_image:.3f} sec")

print("\n----- SEVERITY TASK -----")
print(f"Accuracy       : {acc_severity:.3f}")
print(f"Kappa          : {kappa_severity:.3f}")
print(f"Macro AUC      : {macro_auc_severity:.3f}")
print(f"Inference Time : {inference_time:.3f} sec")
print(f"Time/Image     : {time_per_image:.3f} sec")

print("\n----- COMBINED (AVERAGE) -----")
print(f"Accuracy       : {acc:.3f}")
print(f"Kappa          : {kappa:.3f}")
print(f"Macro AUC      : {macro_auc:.3f}")
print(f"Inference Time : {inference_time:.3f} sec")
print(f"Time/Image     : {time_per_image:.3f} sec")

In [ ]:
import joblib
model.save('best_oct_model.keras')
joblib.dump(disease_encoder, 'disease_encoder.pkl')
joblib.dump(severity_encoder, 'severity_encoder.pkl')
joblib.dump(scaler, 'scaler.pkl')